# Task 2 and Task 3 Analysis Evidence - jzho0172


## Member Scope


In [1]:
from dataclasses import replace

import pandas as pd

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table
from data2001.task4.maps import build_score_choropleth_map, build_poi_density_choropleth_map, build_poi_point_scatter_map

from data2001.task4.queries import (
    load_api_extraction_summary,
    load_correlation_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
)


MEMBER_UNIKEY = "jzho0172"

base_settings = load_settings("configs/local.yaml")
member_sa4 = (base_settings.task2_import.selected_sa4_by_member.get(MEMBER_UNIKEY) or "").strip()
if not member_sa4:
    raise ValueError(f"No SA4 configured for {MEMBER_UNIKEY}. Fill configs/local.yaml selected_sa4_by_member.")

settings = replace(
    base_settings,
    task2_import=replace(
        base_settings.task2_import,
        crawl_scope="selected_sa4",
        selected_sa4_by_member={MEMBER_UNIKEY: member_sa4},
    ),
    task3_score=replace(base_settings.task3_score, score_universe="selected_sa4"),
)
engine = create_engine_from_settings(settings.database)

member_scope = pd.DataFrame([{"unikey": MEMBER_UNIKEY, "selected_sa4": member_sa4}])
display(member_scope)

,unikey,selected_sa4
0,jzho0172,Sydney - North Sydney and Hornsby


## Single-SA4 Full Workflow Run


In [2]:
workflow_steps = [
    "init_db",
    "clear_db",
    "import_boundaries",
    "validate_boundaries",
    "import_poi",
    "import_income",
    "compute_score",
]

workflow_summary = execute_workflow_steps(
    engine,
    settings,
    workflow_steps,
    title=f"{MEMBER_UNIKEY} single-SA4 full rebuild",
)
display(workflow_summary)

{'init_db': 'done',
 'clear_db': 'done',
 'sa4': 1,
 'sa2': 26,
 'population': 2473,
 'boundary_selected_sa4': 1,
 'boundary_sa2_checked': 26,
 'boundary_sa2_valid': 26,
 'boundary_sa2_invalid': 0,
 'boundary_min_coverage_ratio': 0.9999999999999969,
 'boundary_coverage_threshold': 0.999,
 'boundary_point_failures': 0,
 'boundary_missing_parent_sa4': 0,
 'sa2_bbox_requests': 26,
 'raw_responses': 26,
 'raw_features_seen': 5659,
 'clean_features_seen': 3165,
 'fetch_seconds': 4.05004298897984,
 'persist_seconds': 0.06118692000745796,
 'clean_seconds': 0.15998690600099508,
 'load_seconds': 0.044316198996966705,
 'income': 2454,
 'scores': 26,
 'correlations': 2}

## Single-SA4 Database Verification


In [3]:
schema = settings.database.schema_name

display(pd.read_sql(
    f"""
    SELECT sa4_name, COUNT(*) AS sa2_count
    FROM {schema}.sa2
    GROUP BY sa4_name
    """,
    engine,
))

,sa4_name,sa2_count
0,Sydney - North Sydney and Hornsby,26


## Task 2 Evidence: API Extraction and Spatial Join


In [4]:
display(load_api_extraction_summary(settings))
display(load_spatial_join_summary(engine, settings))

,response_dir,response_file_count,features_jsonl,raw_feature_rows,features_file_exists,features_file_size_mb
0,/home/kscii/Codes/data2001-group-assignment/da...,26,/home/kscii/Codes/data2001-group-assignment/da...,5659,True,2.98


,clean_poi,assigned_poi,unassigned_poi,boundary_duplicate_candidates,assignment_rows
0,3165,2352,813,0,2352


## Task 3 Evidence: Score Calculation


In [5]:
display(load_score_input_summary(engine, settings))

scores = load_sa2_scores(engine, settings)
score_map_areas = load_sa2_scores(engine, settings, include_excluded=True)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

,sa2_count,total_poi,mean_poi_count,std_poi_count,min_poi_count,max_poi_count,below_min_population,missing_population
0,26,2352,90.461538,36.593643,36,172,0,0


,sa2_code,sa2_name,sa4_code,sa4_name,population,poi_count,mean_poi_count,std_poi_count,z_poi,score_raw,score_100,is_excluded,exclusion_reason,geometry
0,121011682,Artarmon,121,Sydney - North Sydney and Hornsby,9394,38,90.461538,36.593643,-1.433624,0.192535,19.253459,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
1,121021403,Asquith - Mount Colah,121,Sydney - North Sydney and Hornsby,22370,120,90.461538,36.593643,0.807202,0.691513,69.151295,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
2,121021404,Berowra - Brooklyn - Cowan,121,Sydney - North Sydney and Hornsby,11758,163,90.461538,36.593643,1.982270,0.878923,87.892288,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
3,121011683,Castle Cove - Northbridge,121,Sydney - North Sydney and Hornsby,13336,126,90.461538,36.593643,0.971165,0.725352,72.535163,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
4,121011684,Chatswood - East,121,Sydney - North Sydney and Hornsby,19739,86,90.461538,36.593643,-0.121921,0.469557,46.955742,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."


,rank_group,sa2_code,sa2_name,sa4_name,poi_count,score_100,population
0,top,121031408,Lindfield - Roseville,Sydney - North Sydney and Hornsby,172,90.275467,24661
1,top,121021404,Berowra - Brooklyn - Cowan,Sydney - North Sydney and Hornsby,163,87.892288,11758
2,top,121031407,Gordon - Killara,Sydney - North Sydney and Hornsby,127,73.076200,22435
3,top,121031412,Wahroonga (East) - Warrawee,Sydney - North Sydney and Hornsby,127,73.076200,18150
4,top,121011683,Castle Cove - Northbridge,Sydney - North Sydney and Hornsby,126,72.535163,13336
5,top,121031411,Turramurra,Sydney - North Sydney and Hornsby,124,71.433058,20170
6,top,121021403,Asquith - Mount Colah,Sydney - North Sydney and Hornsby,120,69.151295,22370
7,top,121031410,St Ives,Sydney - North Sydney and Hornsby,119,68.565314,21786
8,top,121041417,North Sydney - Lavender Bay,Sydney - North Sydney and Hornsby,109,62.400988,12651
9,top,121031409,Pymble,Sydney - North Sydney and Hornsby,104,59.145115,17266


## Individual Visual Analysis


In [6]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)
score_income = load_score_income(engine, settings)

### Score Distribution


In [7]:
build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()

### Top and Bottom SA2 Scores


In [8]:
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()

### Score Choropleth Map


In [9]:
build_score_choropleth_map(score_map_areas).show()

### Population-Adjusted POI Density Map


In [10]:
build_poi_density_choropleth_map(score_map_areas).show()


### POI Point Map


In [11]:
build_poi_point_scatter_map(poi_points).show()

### POI Group Distribution


In [12]:
build_poi_group_distribution(poi_groups).show()

### Score and Median Income


In [13]:
build_score_income_scatter(score_income).show()

## Correlation and Interpretation Notes


In [14]:
display(load_correlation_summary(engine, settings))

,method,statistic,p_value,n,alpha,is_significant,created_at,interpretation
0,pearson,-0.143331,0.484851,26,0.05,False,2026-05-20 13:01:17.784705+00:00,not statistically significant
1,spearman,-0.136776,0.505243,26,0.05,False,2026-05-20 13:01:17.784705+00:00,not statistically significant
